In [19]:
import json
import pandas as pd
import numpy as np

# 1. Carregando os dados
with open('../dados/dados_nivel_1.json', 'r', encoding='utf-8') as f:
    dados_brutos = json.load(f)

taxa_cambio = dados_brutos['taxa_cambio_usd_brl']
df = pd.DataFrame(dados_brutos['operacoes'])

print(f"Taxa de Câmbio Fixa: {taxa_cambio}")
print("\n--- Informações do DataFrame ---")
df.info()

print("\n--- Amostra de Valores Únicos nas Colunas Categóricas ---")
print("Moedas:", df['moeda'].unique())
print("Canais:", df['canal'].unique())

print("\n--- Verificando Valores Nulos ou Negativos ---")
display(df[df.isnull().any(axis=1)])
display(df[df['valor'] <= 0])

Taxa de Câmbio Fixa: 5.4

--- Informações do DataFrame ---
<class 'pandas.DataFrame'>
RangeIndex: 20 entries, 0 to 19
Data columns (total 9 columns):
 #   Column       Non-Null Count  Dtype
---  ------       --------------  -----
 0   id           20 non-null     str  
 1   cliente_id   20 non-null     str  
 2   data         19 non-null     str  
 3   valor        20 non-null     int64
 4   moeda        20 non-null     str  
 5   canal        20 non-null     str  
 6   tipo         20 non-null     str  
 7   contraparte  20 non-null     str  
 8   observacao   20 non-null     str  
dtypes: int64(1), str(8)
memory usage: 2.9 KB

--- Amostra de Valores Únicos nas Colunas Categóricas ---
Moedas: <ArrowStringArray>
['BRL', 'USD']
Length: 2, dtype: str
Canais: <ArrowStringArray>
['pix', 'ted', 'boleto', 'cartao', 'especie']
Length: 5, dtype: str

--- Verificando Valores Nulos ou Negativos ---


,id,cliente_id,data,valor,moeda,canal,tipo,contraparte,observacao
17,OP-0017,CLI-A-5,NaN,4300,BRL,especie,deposito,Gama Distribuidora,data nao capturada pelo sistema


,id,cliente_id,data,valor,moeda,canal,tipo,contraparte,observacao


### 2. Limpeza e Tratamento dos Dados

**O que eu encontrei:**  
Dando uma olhada nos dados, vi que tem um valor nulo (`NaN`) na coluna `data`. Isso aconteceu na operação `OP-0017` do cliente `CLI-A-5`. A própria coluna de observação dessa linha diz: "data nao capturada pelo sistema". 

**Como eu resolvi e por quê:**  
Eu decidi **excluir** essa linha usando o `dropna()`. 
Fiz isso porque a Regra 1 (de Fracionamento) pede para somar as operações que acontecem na *mesma data*. Se eu deixar uma operação sem data no DataFrame, o Pandas não vai conseguir agrupar direito e a regra vai falhar ou dar um resultado errado. Como é só uma operação com erro, achei mais seguro tirar ela da análise para não estragar a lógica principal do desafio.

In [20]:
# --- PASSO 2: Limpeza dos Dados ---
# Removendo linhas onde a data é nula, conforme justificado no Markdown
df = df.dropna(subset=['data']).copy()

# Garantindo que a coluna de data seja do tipo datetime para ordenação/agrupamento correto
df['data'] = pd.to_datetime(df['data'])

# --- PASSO 3: Normalização para BRL ---
df['valor_brl'] = np.where(df['moeda'] == 'USD', df['valor'] * taxa_cambio, df['valor'])

# --- PASSO 4: Agregações ---
volume_por_cliente = df.groupby('cliente_id')['valor_brl'].sum().reset_index()
operacoes_por_canal = df['canal'].value_counts().reset_index()

print("--- Volume Total por Cliente ---")
display(volume_por_cliente)
print("\n--- Quantidade de Operações por Canal ---")
display(operacoes_por_canal)

# --- PASSO 5: Implementação das Regras ---

# REGRA 1: Fracionamento 
# (Mesma data, >= 3 ops, soma > 50k, nenhuma >= 20k)
agregado_diario = df.groupby(['cliente_id', 'data'])['valor_brl'].agg(
    soma_diaria='sum', 
    qtd_operacoes='count', 
    maior_operacao='max'
).reset_index()

agregado_diario['flag_fracionamento'] = (
    (agregado_diario['soma_diaria'] > 50000) & 
    (agregado_diario['qtd_operacoes'] >= 3) & 
    (agregado_diario['maior_operacao'] < 20000)
)

# Cruzando a flag de volta para o dataframe original
df = df.merge(agregado_diario[['cliente_id', 'data', 'flag_fracionamento']], on=['cliente_id', 'data'], how='left')

# REGRA 2: Valor Atípico 
# (> 5x mediana do cliente, aplicar apenas a clientes com >= 4 ops)
stats_cliente = df.groupby('cliente_id')['valor_brl'].agg(
    mediana_cliente='median', 
    qtd_total_ops='count'
).reset_index()

df = df.merge(stats_cliente, on='cliente_id', how='left')

df['flag_atipico'] = (
    (df['valor_brl'] > (df['mediana_cliente'] * 5)) & 
    (df['qtd_total_ops'] >= 4)
)

# Limpando colunas auxiliares usadas na regra 2
df = df.drop(columns=['mediana_cliente', 'qtd_total_ops'])

# --- PASSO 6: Validação das Regras ---
print("\n--- Validação da Regra 1 (Fracionamento) ---")
# Filtramos apenas os clientes que tiveram movimentações no dia 2026-03-09 para ver o comportamento
validacao_r1 = df[df['data'] == '2026-03-09'][['cliente_id', 'valor_brl', 'flag_fracionamento']].sort_values('cliente_id')
display(validacao_r1)

--- Volume Total por Cliente ---


,cliente_id,valor_brl
0,CLI-A-1,57500.0
1,CLI-A-2,52900.0
2,CLI-A-3,65700.0
3,CLI-A-4,79500.0
4,CLI-A-5,12600.0
5,CLI-A-6,10200.0



--- Quantidade de Operações por Canal ---


,canal,count
0,pix,9
1,ted,5
2,boleto,3
3,cartao,2



--- Validação da Regra 1 (Fracionamento) ---


,cliente_id,valor_brl,flag_fracionamento
0,CLI-A-1,18100.0,True
1,CLI-A-1,17300.0,True
2,CLI-A-1,18800.0,True


In [22]:
import os
import time
import json
import google.generativeai as genai
from dotenv import load_dotenv
from IPython.display import display, Markdown

# 1. Carregar a chave de API do arquivo .env
load_dotenv()
CHAVE_API = os.getenv("GEMINI_API_KEY")

if not CHAVE_API:
    print("ERRO: Chave API não encontrada! Verifique seu arquivo .env")
else:
    genai.configure(api_key=CHAVE_API)
    modelo_ia = genai.GenerativeModel('gemini-3.6-flash')
    
    # ==========================================================================
    # ETAPA RAG 1: Base de Conhecimento Regulatório (Knowledge Base)
    # ==========================================================================
    BASE_CONHECIMENTO_PLD = {
        "fracionamento": (
            "Carta Circular BACEN nº 4.001/2020, Art. 2º, Inciso II: "
            "Exemplifica como operação suspeita a realização de operações sucessivas ou fracionadas, "
            "em espécie ou por meio de transferências (PIX/TED), com valores individuais inferiores "
            "ao limite regulatório de comunicação (R$ 50.000,00), mas cujo montante acumulado "
            "configure tentativa de burlar os mecanismos de controle e reporte ao COAF (Structuring)."
        ),
        "valor_atipico": (
            "Circular BACEN nº 3.978/2020, Art. 38 c/c Lei nº 9.613/1998, Art. 11: "
            "Instituições financeiras devem monitorar transações com valores desproporcionais ou "
            "incompatíveis com o padrão histórico, volume e capacidade financeira usual do cliente."
        )
    }

    try:
        # 3. Filtrar o cliente suspeito (CLI-A-1)
        cliente_suspeito = "CLI-A-1"
        dados_cli_a1 = df[df['cliente_id'] == cliente_suspeito].copy()
        dados_cli_a1['data'] = dados_cli_a1['data'].dt.strftime('%Y-%m-%d')
        extrato_cliente = dados_cli_a1.to_dict(orient='records')
        
        # ======================================================================
        # ETAPA RAG 2: Retrieval (Recuperação da norma vinculada à flag ativada)
        # ======================================================================
        flag_ativa = "fracionamento" if dados_cli_a1['flag_fracionamento'].any() else "valor_atipico"
        contexto_regulatorio = BASE_CONHECIMENTO_PLD.get(flag_ativa, "Normativas gerais de PLD-FT.")

        # ======================================================================
        # ETAPA RAG 3: Augmentation (Injeção do contexto legal no Prompt)
        # ======================================================================
        prompt = f"""
        Você é um analista sênior de Prevenção à Lavagem de Dinheiro (PLD).
        Analise o extrato do cliente {cliente_suspeito}:
        {json.dumps(extrato_cliente, indent=2)}

        --- BASE NORMATIVA RECUPERADA (RAG) ---
        {contexto_regulatorio}

        Instruções:
        - Utilize a base normativa acima para fundamentar tecnicamente a análise.
        - Devolva a análise ESTRITAMENTE no formato JSON abaixo, sem blocos de texto adicionais:
        {{
          "nivel_risco": "baixo, medio ou alto",
          "tipologia_suspeita": "texto curto",
          "red_flags": ["lista de alertas objetivos"],
          "fundamentacao_legal": "artigo/norma recuperada aplicada ao caso",
          "justificativa": "parecer técnico completo integrando os dados e a norma"
        }}
        """
        
        print("Iniciando análise com RAG regulatório e modelo gemini-3.6-flash...\n")
        inicio = time.time()
        
        # Chamando a IA
        resposta = modelo_ia.generate_content(prompt)
        tempo_gasto = time.time() - inicio
        
        # Sanitização do JSON
        texto_resposta = resposta.text.strip()
        if texto_resposta.startswith("```json"):
            texto_resposta = texto_resposta[7:-3]
        elif texto_resposta.startswith("```"):
            texto_resposta = texto_resposta[3:-3]
        
        # Parse e Exibição Formatada
        try:
            analise_json = json.loads(texto_resposta)
            
            nivel = analise_json.get('nivel_risco', '').lower()
            icone_risco = "🔴" if nivel == "alto" else "🟡" if nivel == "medio" else "🟢"
            
            relatorio = f"""
### 🚨 Relatório de Parecer PLD com RAG - Cliente: `{cliente_suspeito}`

**Nível de Risco:** {icone_risco} **{nivel.upper()}**  
**Tipologia Suspeita:** {analise_json.get('tipologia_suspeita')}  
**Base Legal:** `{analise_json.get('fundamentacao_legal', 'BACEN/COAF')}`

#### 🚩 Red Flags Identificadas:
"""
            for flag in analise_json.get('red_flags', []):
                relatorio += f"- {flag}\n"
                
            relatorio += f"""
#### 📋 Parecer e Fundamentação Técnica:
*{analise_json.get('justificativa')}*

---
*⏱️ Tempo de resposta: {tempo_gasto:.2f} segundos* | *🪙 Tokens (In: {resposta.usage_metadata.prompt_token_count} | Out: {resposta.usage_metadata.candidates_token_count} | Total: {resposta.usage_metadata.total_token_count})*
"""
            display(Markdown(relatorio))
            
        except json.JSONDecodeError:
            print("Erro: A IA não devolveu um JSON válido.")
            print("Resposta bruta:", texto_resposta)
            
    except NameError:
        print("\n❌ ERRO: A tabela 'df' não foi encontrada na memória.")
        print("👉 SOLUÇÃO: Execute a célula anterior com os dados do Pandas.")

Iniciando análise com RAG regulatório e modelo gemini-3.6-flash...




### 🚨 Relatório de Parecer PLD com RAG - Cliente: `CLI-A-1`

**Nível de Risco:** 🔴 **ALTO**  
**Tipologia Suspeita:** Fracionamento de operações (Structuring) para eversão de limites de reporte ao COAF  
**Base Legal:** `Carta Circular BACEN nº 4.001/2020, Art. 2º, Inciso II`

#### 🚩 Red Flags Identificadas:
- Realização de três transferências no mesmo dia (09/03/2026) somando R$ 54.200,00, montante que supera o limite regulatório de comunicação (R$ 50.000,00).
- Fracionamento de valores individuais abaixo do limite de alerta regulatório (operações de R$ 17.300, R$ 18.100 e R$ 18.800).
- Múltiplos pagamentos via PIX no mesmo dia para a mesma contraparte (Alfa Comercio LTDA), totalizando R$ 35.400,00.
- Presença de flag de fracionamento ativada (flag_fracionamento: true) em três operações consecutivas.

#### 📋 Parecer e Fundamentação Técnica:
*O cliente CLI-A-1 realizou, no dia 09/03/2026, três movimentações financeiras de saída (duas via PIX e uma via TED) cujos valores individuais variaram entre R$ 17.300,00 e R$ 18.800,00, totalizando R$ 54.200,00 no mesmo dia. O padrão comportamental de fracionar valores individuais mantendo-os abaixo da alçada regulatória de R$ 50.000,00, mas com montante acumulado superior a esse patamar — aliado ao envio sequencial de valores para a mesma contraparte (Alfa Comercio LTDA) —, enquadra-se rigorosamente na hipótese de 'Structuring' prevista na Carta Circular BACEN nº 4.001/2020, Art. 2º, Inciso II. A conduta evidencia tentativa deliberada de burlar os mecanismos automáticos de monitoramento e reporte ao COAF, justificando a elevação do risco para ALTO e a recomendação de comunicação ao órgão regulador.*

---
*⏱️ Tempo de resposta: 20.61 segundos* | *🪙 Tokens (In: 898 | Out: 531 | Total: 2644)*


### 💡 Avaliação do Parecer Gerado

* **Precisão Fática:** A LLM não alucinou valores nem datas; todos os números citados no parecer técnico coincidem estritamente com as agregações prévias do Pandas.
* **Qualidade Regulatória:** Identificação correta da tipologia de *Smurfing/Structuring*, contextualizando o fracionamento intencional abaixo da régua de R$ 50 mil.
* **Prontidão para Produção:** O retorno estrito em JSON viabiliza o consumo do parecer por filas de mensageria, sistemas legados de investigação ou envio automatizado de Comunicações de Operações Suspeitas (COS) ao COAF.